# WASP-8: cleaning and a residual transit search

Download a TESS light curve, remove non-finite measurements, apply manual quality cuts, normalize and flatten the flux, mask known transits, and search the residuals with Box Least Squares. Inspect the masks and periodogram before interpreting any peak. This is an exploratory search, not evidence for an additional planet.

## Before running
Run from the repository root after installing `requirements.txt`. Archive access requires internet. Inspect the Lightkurve search table and explicitly choose the intended sector/product: numeric result indices are not stable archive identifiers. Saved execution outputs are intentionally absent.

## Review the results
Compare raw and cleaned flux; inspect masked regions and model residuals. Record the selected data product, assumptions, and uncertainties before using a result in scientific work.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import lightkurve as lk
from mandelagol import occultquad
import lmfit as lmfit
from scipy.stats import chi2
from astropy.timeseries import BoxLeastSquares

plt.rcParams['font.size'] = 18
plt.rcParams['figure.figsize'] = [20,10]

In [ ]:
def tlc(params, t):

    tc = params['tc']
    period = params['period']
    u = params['u']
    k = params['k']
    b = params['b']
    a = params['a']
    flux_oot = params['flux_oot']
    dflux_dt = params['dflux_dt']

    '''
    
    Computes loss of light due to a transiting planet using formulas from Mandel & Agol (2002)
    and assuming linear limb-darkening law.

    Inputs:
     t0 = midpoint of transit
      k = (radius of planet) / (radius of star)
      a = (radius of orbit) / (radius of star)
      b = transit impact parameter = a*cos(incl), should be <= 1
      u = limb darkening parameter (0.0 for uniform intensity, 0.6 for Sun-like)
      flux_oot = value of flux outside of transits
      dflux_dt = fractional rate of change of flux_oot per day
      
    Outputs:
      calculated flux, with flux=1 outside of transit.
      
    '''
    
    phi = 2.*np.pi*(t-tc)/period  # the phase of the orbit
    x = a*np.sin(phi)             # the x-coordinate with reference to Fig. 2 of Winn (2010)
    y = b*np.cos(phi)             # the y-coordinate with reference to Fig. 2 of Winn (2010)
    z = np.sqrt(a*a-b*b)*np.cos(phi)
    s = np.sqrt(x*x + y*y)        # the distance between planet and star on the sky
    flux_calc, flux_no_limb_darkening = occultquad(s,u,0.0,k) # calculates the loss of light
    flux_calc[z<0.0] = 1.0        # dip occurs only when planet is in front of star (not behind)
    
    flux_calc_with_slope = flux_calc * flux_oot * (1.0 + dflux_dt * (t-tc)) 
    
    return(flux_calc_with_slope)

def tlc_residuals(params, t, f, f_unc):
    res = (f - tlc(params, t))/f_unc
    return(res)

def fold_time(t, tc_ref, period_ref):
    
    '''
    
    Converts time into "folded time" in the range between -period/2 and +period/2
    as well as an orbit number, assigning tc_ref = orbit zero
     
    '''  
    
    orbit_number, t_fold = np.divmod( t - (tc_ref - 0.5*period_ref), period_ref )
    orbit_number = orbit_number.astype(int)
    
    t_fold = t_fold - 0.5*period_ref
 
    return(t_fold, orbit_number)

def chisqr_pval(chisqr, ndof):
    
    '''
    
    inputs are chi-squared of best-fit model and number of degrees of freedom

    output is the probability that such a large chi-squared would be observed
    if the model is correct and the input data have independent Gaussian uncertainties
    
    '''
    
    return(1.0 - chi2.cdf(chisqr,ndof))

def remove_time_intervals(t, f, time_ranges):
    
    '''
    
    Removes specified time ranges from a light curve (because you see
    some obviously bad data, such as at the start and end of a TESS light curve)

    t = original time
    f = original flux
    time_ranges = a list such as np.array([t1,t2], [t3,t4], [t5,t6])
          which would cause this routine to delete the data between
          times t1-->t2, times t3-->t4, and times t5-->t6.

    '''  

    for time_range in time_ranges:
        t1 = time_range[0]
        t2 = time_range[1]
        bad = (t > t1) & (t < t2)
        t = t[~bad]
        f = f[~bad]
    return(t, f)

def flatten_lightcurve(t, f, t_mask, time_interval, polynomial_order):

    '''
    
    Filter out slow variations in a light curve by fitting polynomial
    functions of time. Also called "Savitsky-Golay" filtering (I think)
    
    t = original time
    f = original flux
    t_mask = Boolean array specifying which times should NOT be used in
             the fit.  For example, times that are near transits.
    time_interval = for each data point t[i], the fit will be performed on the
                    data spanning time_interval centered on t[i].
                    Choose based on timescale of variations.
                    Should probably always be longer than 1 day to avoid
                    overfitting. 
    polynomial_order = order of polynomial, obviously. Should probably always
                    be 1, 2, or 3 to avoid overfitting.
                    

    '''  

    
    n_data = len(t)
    f_flat = np.zeros(n_data)
    for i in range(n_data):
        fit = (t >= t[i] - 0.5*time_interval) & (t < t[i] + 0.5*time_interval) & (~t_mask)
        if np.sum(fit) > 10 + polynomial_order:
            t_fit = t[fit]
            f_fit = f[fit]
            coeff = np.polyfit(t_fit, f_fit, polynomial_order)
            f_flat[i] = f[i] / np.polyval(coeff, t[i])
        else:
            f_flat[i] = f[i]
    return(f_flat)

In [ ]:
search_result = lk.search_lightcurve('WASP-8', mission='TESS')
search_result

In [ ]:
# To avoid abbreviation of the table, use this command
search_result.table[['#','mission','author','exptime']].pprint(100)

In [ ]:
lc = search_result[6].download().remove_nans()

time_entire = np.array(lc.time.to_value('jd'))
flux_entire = np.array(lc.flux.to_value())

bjd_offset = 2457000.0
time_entire = time_entire - bjd_offset

In [ ]:
plt.plot(time_entire, flux_entire, 'o', alpha=0.1)
plt.xlabel('Time (BTJD)')
plt.ylabel('Flux (e/s)')
plt.title('Straight from the archive')
plt.grid()

In [ ]:
time_intervals = [[3187.5, 3189], [3200,3202], [3205,3207]] # data in these time ranges look bad
time, flux = remove_time_intervals(time_entire, flux_entire, time_intervals)
plt.plot(time, flux, 'o', alpha=0.1)
plt.xlabel('Time (BTJD)')
plt.ylabel('Flux (e/s)')
plt.title('After omitting bad data')
plt.grid()

In [ ]:
tc_ref = 2458358.92067 - bjd_offset # from Ivshina & Winn (2022)
period_ref = 8.1587289             # from Ivshina & Winn (2022)
tra_dur_ref = 4.4/24.             # from Queloz et al. (2010)

time_folded, orbit_number = fold_time(time, tc_ref, period_ref)
near_transit = np.abs(time_folded) < (2.0 * tra_dur_ref)
flux = flux / np.median(flux[~near_transit]) # normalize light curve

fig, (ax1, ax2, ax3) = plt.subplots(3,1,figsize=(20,10))

ax1.plot(time, flux, '.', alpha=0.3)
ax1.plot(time[near_transit], flux[near_transit], 'r.')
ax1.set_xlabel('Time (BTJD)')
ax1.set_ylabel('Flux')
ax1.set_title('Normalized light curve')

ax2.plot(time, orbit_number, '.')
ax2.plot(time[near_transit], orbit_number[near_transit], 'r.')
ax2.set_xlabel('Time (BTJD)')
ax2.set_ylabel('Orbit number')
ax2.set_title('Orbit number of each data point')

ax3.plot(time_folded, flux, '.')
ax3.plot(time_folded[near_transit], flux[near_transit], 'r.')
ax3.set_xlabel('Folded Time (days)')
ax3.set_ylabel('Flux')
ax3.set_title('Normalized light curve vs. folded time')

plt.tight_layout()

In [ ]:
# There are slow variations, due to stellar rotation.
# Let's filter them out by fitting quadratic functions to each 3-day time interval,
# but we should not include the hot-Jupiter transits in the fits. We will
# use the "near_transit" array to specify which points to exclude from the fits.

flux_flat = flatten_lightcurve(time, flux, near_transit, 3, 2)

# (this step takes a while because it operates on every single data point)

In [ ]:
plt.plot(time, flux_flat, 'o', alpha=0.1)
plt.plot(time[near_transit], flux_flat[near_transit], 'ro', alpha=0.1)
plt.xlabel('Time (BTJD)')
plt.ylabel('Flux')
plt.title('After filtering out slow variations')

In [ ]:
# Now let's omit the hot-Jupiter transits, to prepare to conduct a BLS search
# for smaller planets that we cannot see by eye.

t = time[~near_transit]
f = flux_flat[~near_transit]
plt.plot(t, f, 'o', alpha=0.1)
plt.xlabel('Time (BTJD)')
plt.ylabel('Flux')
plt.title('After omitting Hot Jupiter transits')

In [ ]:
# Let's perform a BLS search, using 1000 trial periods between 0.3 and 10 days,
# and 8 trial durations ranging from 15 minutes to 4 hours.
periods = np.linspace(0.3, 10, 1000)
durations = np.linspace(0.25,4,8)/24.0
bls = BoxLeastSquares(t=t, y=f, dy=np.std(f))
results = bls.power(periods, durations)

# This command triggers a WARNING but we can ignore it for now.

In [ ]:
# Plot the periodogram

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.plot(results.period, results.power, "ko-", lw=0.5, markersize=2)
ax.set_xlim(results.period.min(), results.period.max())
ax.set_xlabel("Period (days)")
ax.set_ylabel("Log(Likelihood) of Transits")
ax.grid()

# Find the period of the peak
period_peak = results.period[np.argmax(results.power)]
print('Max BLS power is at period = ', period_peak)

# In this case, there is no peak that clearly stands out above the rest,
# so we do not have any good evidence for an additional planet in the
# requested period range.